# Random Forest modell – Machine Learning beadandó

## Modell relevanciája

A Random Forest egy ensemble machine learning algoritmus, amely több döntési fát kombinál annak érdekében, hogy pontosabb és stabilabb predikciót készítsen.

A modell működése:

- több döntési fát tanít különböző mintákon,
- az egyes fák szavaznak,
- a végső predikció a többségi döntés alapján születik.

Előnyei:

- jobb generalizáció,
- kisebb overfitting kockázat,
- stabilabb teljesítmény,
- feature importance meghatározása.

A Random Forest gyakran jobb eredményt ad, mint egyetlen Decision Tree modell.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)


## 1. Train és test csv beolvasás

In [ ]:
train_df = pd.read_csv("../../DataCleaning/train.csv")
test_df = pd.read_csv("../../DataCleaning/test.csv")

## 2. X és y szétválasztása

In [ ]:
target_col = "target"

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## 3. Feature kiválasztás

In [ ]:
numeric_columns = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

binary_features = ["type_L", "type_M"]

numeric_features = [
    col for col in numeric_columns
    if col not in binary_features
]

selected_features = numeric_features + binary_features

print("Feature count:", len(selected_features))

## 4. Pipeline létrehozása

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("features", "passthrough", selected_features)
    ],
    remainder="drop"
)

random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_estimators=200,
            max_depth=8,
            min_samples_split=5,
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])

random_forest_pipeline

## 5. Keresztvalidáció

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    estimator=random_forest_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=True
)

cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Train mean": [
        cv_results["train_accuracy"].mean(),
        cv_results["train_precision"].mean(),
        cv_results["train_recall"].mean(),
        cv_results["train_f1"].mean(),
        cv_results["train_roc_auc"].mean()
    ],
    "Validation mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean()
    ]
})

cv_summary = cv_summary.round(4)

cv_summary

## 6. Hyperparameter tuning

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 8, 12],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 3]
}

grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

## 7. Legjobb paraméterek

In [ ]:
print("Legjobb paraméterek:")
print(grid_search.best_params_)

print("\nLegjobb F1-score:")
print(round(grid_search.best_score_, 4))

## 8. Final model

In [ ]:
final_random_forest_model = grid_search.best_estimator_

final_random_forest_model

## 9. Predikció

In [ ]:
y_pred = final_random_forest_model.predict(X_test)

y_prob = final_random_forest_model.predict_proba(X_test)[:, 1]

## 10. Modell kiértékelése

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

results_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

results_df = results_df.round(4)

results_df

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm
).plot(ax=ax)

plt.title("Random Forest - Confusion Matrix")
plt.show()

## 12. Classification Report

In [ ]:
print(classification_report(y_test, y_pred))

## 13. ROC görbe

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)

plt.title("Random Forest - ROC Curve")
plt.show()

## 14. Threshold comparison

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:

    y_threshold_pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_threshold_pred),
        "Precision": precision_score(y_test, y_threshold_pred),
        "Recall": recall_score(y_test, y_threshold_pred),
        "F1-score": f1_score(y_test, y_threshold_pred)
    })

threshold_comparison_df = pd.DataFrame(threshold_results)

threshold_comparison_df = threshold_comparison_df.round(4)

threshold_comparison_df

## 15. Feature importance

In [ ]:
model = final_random_forest_model.named_steps["model"]

feature_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

feature_importance_df.head(15)

## 16. Feature importance vizualizáció

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance_df["Feature"][:10][::-1],
    feature_importance_df["Importance"][:10][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importance - Random Forest")

plt.show()

## 17. Predikciók exportálása

In [ ]:
prediction_export_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability": y_prob
})

prediction_export_df.head()

## 18. CSV exportok

In [ ]:
cv_summary.to_csv(
    "random_forest_cv_summary.csv",
    index=False
)

results_df.to_csv(
    "random_forest_final_results.csv",
    index=False
)

threshold_comparison_df.to_csv(
    "random_forest_threshold_comparison.csv",
    index=False
)

feature_importance_df.to_csv(
    "random_forest_feature_importance.csv",
    index=False
)

prediction_export_df.to_csv(
    "random_forest_predictions.csv",
    index=False
)

print("Minden CSV export sikeresen elkészült.")